# ONG v3 — Temperature scaling (calibration) for all four backbones

Fits a single post-hoc temperature `T` on **`val_live.csv`** (LBFGS, NLL; Guo et al. 2017) and reports **test ECE before -> after** on the frozen split for DINOv2, BioCLIP 2, ConvNeXt V2-L and EfficientNetV2-L. Top-1 / macro accuracy are invariant to `T`, so only calibration changes. Produces the `ECE (post-TS)` column for Table 2.

**Before running:** upload the patched `scripts/13_evaluate.py` (with `--fit-temperature`) to Drive, and confirm `models/<key>/best_model_global.pth` + `vocab.json` exist.

Runtime: any GPU (L4/T4 fine — this is inference only, ~10-20 min total).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/ONG_v3'
SCRIPTS = ROOT + '/scripts'
!pip -q install timm open_clip_torch
import os; print('scripts:', os.listdir(SCRIPTS) if os.path.isdir(SCRIPTS) else 'MISSING')


In [ ]:
import os
if not os.path.isdir('/content/photos') or len(os.listdir('/content/photos')) < 50:
    print('Unzipping photos.zip ...')
    !cd /content && unzip -q -o {ROOT}/photos.zip -d /content/_pz
    # the zip root may be 'photos/' or the genus dirs directly
    import glob, shutil
    cand = '/content/_pz/photos' if os.path.isdir('/content/_pz/photos') else '/content/_pz'
    if cand != '/content/photos':
        shutil.move(cand, '/content/photos') if not os.path.isdir('/content/photos') else None
    print('photos genera:', len(os.listdir('/content/photos')))
else:
    print('photos already present:', len(os.listdir('/content/photos')), 'genera')


In [ ]:
import os, subprocess

# (key, arch, model name, img_size)
MODELS = [
    ('dinov2l',     'timm',     'vit_large_patch14_reg4_dinov2.lvd142m', 448),
    ('bioclip2',    'openclip', 'hf-hub:imageomics/bioclip-2',           224),
    ('convnextv2l', 'timm',     'convnextv2_large.fcmae_ft_in22k_in1k',  384),
    ('effnetv2l',   'timm',     'tf_efficientnetv2_l.in21k_ft_in1k',     448),
]

def ckpt_for(key):
    g = f'{ROOT}/models/{key}/best_model_global.pth'   # paper Table 2 used the global-select ckpt
    b = f'{ROOT}/models/{key}/best_model.pth'
    return g if os.path.exists(g) else b

for key, arch, name, sz in MODELS:
    ck = ckpt_for(key); vocab = f'{ROOT}/models/{key}/vocab.json'
    if not (os.path.exists(ck) and os.path.exists(vocab)):
        print(f'!! SKIP {key}: missing checkpoint or vocab ({ck})'); continue
    print(f'\n===== {key}  ({name} @ {sz})  ckpt={os.path.basename(ck)} =====')
    cmd = ['python', f'{SCRIPTS}/13_evaluate.py', '--arch', arch, '--model', name,
           '--img-size', str(sz), '--checkpoint', ck, '--vocab', vocab,
           '--no-retrieval', '--bootstrap', '0', '--fit-temperature',
           '--photos-root', '/content/photos', '--out-dir', f'{ROOT}/eval_ts/{key}']
    subprocess.run(cmd, check=False)


In [ ]:
import json, os, datetime
rows = []
order = ['dinov2l', 'bioclip2', 'convnextv2l', 'effnetv2l']
label = {'dinov2l':'DINOv2 ViT-L/14','bioclip2':'BioCLIP 2 ViT-L/14',
         'convnextv2l':'ConvNeXt V2-L','effnetv2l':'EfficientNetV2-L'}
for key in order:
    p = f'{ROOT}/eval_ts/{key}/results.json'
    if not os.path.exists(p): continue
    c = json.load(open(p))['classification']
    rows.append((label[key], c.get('temperature'), c.get('ece'), c.get('ece_post_ts')))

lines = ['| Backbone | T* | ECE (raw) | ECE (post-TS) |',
         '|----------|----|-----------|---------------|']
for name, T, e0, e1 in rows:
    lines.append(f'| {name} | {T:.3f} | {e0:.3f} | {e1:.3f} |')
table = '\n'.join(lines)
print(table)

os.makedirs(f'{ROOT}/eval_ts', exist_ok=True)
out = f'{ROOT}/eval_ts/temperature_summary_' + datetime.date.today().isoformat() + '.md'
open(out, 'w').write('# Temperature scaling (test ECE before/after)\n\n' + table + '\n')
print('\nSaved ->', out)
